# MultiWorld Causal Subsidy — End-to-End Demo

> Causal inference + AI-driven simulation for coupon subsidy policy evaluation.
> Originated from the 6th Meituan Business Analysis Elite Competition.

This notebook demonstrates the full pipeline:
1. **Data Generation** — Synthetic data mimicking Meituan coupon/order structure
2. **Causal Inference** — CausalML (T/X/DR/S-Learner) + DoWhy causal graph
3. **Theory-Driven Agents** — Prospect Theory + Mental Accounting
4. **Mesa ABM Simulation** — Agent-based model with cognitive agents
5. **Multi-World Evaluation** — Parallel worlds with different subsidy strategies
6. **Network Contagion** — Social network spillover effects
7. **Evaluation Metrics** — Bootstrap CI, E-value, robustness


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fix Chinese font display (方块/方框 issue)
plt.rcParams['font.sans-serif'] = ['PingFang HK', 'Heiti TC', 'Arial Unicode MS', 'SimHei', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False  # Fix minus sign display

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Environment ready. Chinese font configured.')

Environment ready.


## 1. Data Generation

We generate synthetic data that mimics the structure of the Meituan competition dataset (anonymized). The causal data includes a known true ATE for validating our estimators.

In [2]:
from src.features.data_generator import generate_all_data, SyntheticDataConfig

config = SyntheticDataConfig(n_users=5000, n_orders=20000)
data = generate_all_data(config)

user_profiles = data['user_profiles']
orders = data['orders']
causal_data = data['causal_data']

print(f'User profiles: {user_profiles.shape}')
print(f'Orders: {orders.shape}')
print(f'Causal data: {causal_data.shape}')
print(f'\nTrue ATE in causal data: {causal_data["true_cate"].mean():.4f}')
print(f'Treatment rate: {causal_data["treatment"].mean():.2%}')
causal_data.head()


User profiles: (5000, 21)
Orders: (16321, 7)
Causal data: (5000, 23)

True ATE in causal data: 2.0075
Treatment rate: 49.52%


,Z1,Z2,Z3,Z4,treatment,outcome,feature_0,feature_1,feature_2,feature_3,...,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,true_cate
0,0.496714,0,-0.938467,4,0,-0.510847,-0.294658,0.702143,1.100380,0.115537,...,-1.175990,-1.162523,-1.501653,-1.163548,-1.052987,-0.417969,0.255433,-0.258204,0.658717,1.933634
1,-0.138264,0,-0.546408,5,0,-1.850801,1.449390,0.047287,-1.376582,-1.003769,...,-2.235134,-1.382733,-1.748714,0.358689,-0.689097,-0.132775,0.521625,-0.760354,1.610729,1.533891
2,0.647689,0,-0.282241,3,1,4.639877,0.932251,1.402898,-1.108629,-0.533732,...,-0.703181,-0.348922,1.018398,-0.996199,1.115796,0.239446,0.570629,-0.379654,0.622713,2.478344
3,1.523030,0,1.050774,3,0,1.698942,-0.218599,-0.608116,0.522416,0.276792,...,-0.014134,1.424884,-0.357720,-0.041262,1.414012,0.665305,0.768796,0.893279,0.162394,4.153494
4,-0.234153,0,-1.162939,2,0,-1.238436,1.245585,-1.236379,-1.165369,-1.080680,...,1.230297,0.993794,1.425628,0.685347,-0.659822,-0.255810,-0.329413,-0.699726,-1.233968,1.068083


## 2. Causal Inference

### 2.1 CausalML — Meta-Learner Comparison

We compare four Meta-Learner approaches (T/X/DR/S-Learner) for estimating Conditional Average Treatment Effects (CATE).

In [3]:
from src.modeling.causalml_wrapper import CausalMLWrapper, CausalMLConfig

feature_cols = [c for c in causal_data.columns if c not in ('treatment', 'outcome', 'true_cate')]
true_cate = causal_data['true_cate'].values

learner_results = {}
for learner_type in ['tlearner', 'xlearner', 'drlearner', 'slearner']:
    config = CausalMLConfig(learner_type=learner_type)
    wrapper = CausalMLWrapper(config)
    result = wrapper.fit_predict(causal_data, feature_cols, 'treatment', 'outcome')
    cate = result['cate_causalml'].values.flatten()
    corr = np.corrcoef(cate, true_cate)[0, 1]
    learner_results[learner_type] = {'ate': cate.mean(), 'corr': corr, 'cate': cate}
    print(f'{learner_type.upper():12s} | ATE={cate.mean():.4f} | Corr with true CATE={corr:.4f}')


TLEARNER     | ATE=2.0572 | Corr with true CATE=0.8458


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


XLEARNER     | ATE=2.0053 | Corr with true CATE=0.9408


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear

DRLEARNER    | ATE=-2.2411 | Corr with true CATE=0.2190
SLEARNER     | ATE=2.0562 | Corr with true CATE=0.8470


In [4]:
# Visualize CATE distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CATE distributions
for name, res in learner_results.items():
    axes[0].hist(res['cate'], bins=50, alpha=0.5, label=name.upper())
true_cate_arr = causal_data['true_cate'].values
axes[0].hist(true_cate_arr, bins=50, alpha=0.3, label='True CATE', color='black')
axes[0].set_xlabel('CATE')
axes[0].set_ylabel('Count')
axes[0].set_title('CATE Distributions by Learner')
axes[0].legend()

# Correlation with true CATE
names = list(learner_results.keys())
corrs = [learner_results[n]['corr'] for n in names]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
bars = axes[1].bar([n.upper() for n in names], corrs, color=colors)
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect correlation')
axes[1].set_ylabel('Correlation with True CATE')
axes[1].set_title('Learner Accuracy Comparison')
axes[1].legend()

plt.tight_layout()
plt.savefig('../output/notebook_cate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.2 DoWhy — Causal Graph & Refutation

DoWhy constructs a causal DAG and provides refutation tests to validate the estimated ATE.

In [5]:
from src.modeling.dowhy_causal_graph import SubsidyCausalGraph, DoWhyConfig

config = DoWhyConfig()
cg = SubsidyCausalGraph(config)

# Estimate ATE
ate_result = cg.estimate_ate(causal_data, 'treatment', 'outcome')
print(f'DoWhy ATE = {ate_result.get("ate", "N/A")}')
print(f'True ATE = {true_cate.mean():.4f}')
print(f'Method: {ate_result.get("method", "N/A")}')
if 'ate_lower_ci' in ate_result:
    print(f'95% CI: [{ate_result["ate_lower_ci"]:.4f}, {ate_result["ate_upper_ci"]:.4f}]')


DoWhy ATE = 2.0618351404724433
True ATE = 2.0075
Method: backdoor.propensity_score_matching
95% CI: [1.9809, 2.1556]


In [6]:
# Refutation tests
# Note: refutation_test() uses the saved model state from estimate_ate() above
refute_result = cg.refutation_test()
for method, res in refute_result.items():
    new_eff = res.get('new_effect', 'N/A')
    if isinstance(new_eff, (int, float)):
        print(f'  {method}: new_effect={new_eff:.4f}')
    else:
        print(f'  {method}: new_effect={new_eff}')

AttributeError: 'SubsidyCausalGraph' object has no attribute 'refute_estimate'

## 3. Theory-Driven Cognitive Agents

Our agents are grounded in behavioral economics:
- **Prospect Theory** (Kahneman & Tversky, 1979): Loss aversion with λ=2.25
- **Mental Accounting** (Thaler, 1985): Different reference point update rates by account type
- **Bounded Rationality** (Simon, 1955): Finite decision precision

In [ ]:
from src.simulation.cognitive_agent_theory import (
    TheoreticalCognitiveAgent, MentalAccountType,
    prospect_value, prospect_discount
)

# Demonstrate Prospect Theory value function
x = np.linspace(-20, 20, 200)
v_gain = np.array([prospect_value(xi) for xi in x if xi >= 0])
v_loss = np.array([prospect_value(xi) for xi in x if xi < 0])
x_gain = np.array([xi for xi in x if xi >= 0])
x_loss = np.array([xi for xi in x if xi < 0])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_gain, v_gain, 'b-', linewidth=2, label='Gains (concave)')
ax.plot(x_loss, v_loss, 'r-', linewidth=2, label='Losses (convex, steeper)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Subsidy amount (¥)')
ax.set_ylabel('Subjective value')
ax.set_title('Prospect Theory Value Function\n(α=0.88, λ=2.25; Kahneman & Tversky, 1979)')
ax.legend()
plt.tight_layout()
plt.savefig('../output/notebook_prospect_theory.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Simulate different mental account types over 8 rounds
np.random.seed(42)
account_results = {}

for account_type in MentalAccountType:
    agent = TheoreticalCognitiveAgent(
        agent_id=f'demo_{account_type.value}',
        mental_account=account_type,
        price_sensitivity=0.5,
    )
    redemptions = []
    for r in range(8):
        decided = agent.decide(subsidy_amount=10.0)
        agent.update_state(was_subsidized=True, redeemed=decided)
        redemptions.append(1 if decided else 0)
    account_results[account_type.value] = redemptions

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
rounds = range(1, 9)
for name, redemptions in account_results.items():
    ax.plot(rounds, np.cumsum(redemptions), 'o-', label=name)
ax.set_xlabel('Round')
ax.set_ylabel('Cumulative Redemptions')
ax.set_title('Redemption Behavior by Mental Account Type\n(8 rounds, ¥10 subsidy each)')
ax.legend()
ax.set_xticks(list(rounds))
plt.tight_layout()
plt.savefig('../output/notebook_mental_accounts.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Mesa ABM Simulation

Using the Mesa framework for formal agent-based modeling. Each agent incorporates the behavioral economics model from above.

In [ ]:
from src.simulation.mesa_agent_model import SubsidyModel

# Run cognitive strategy for 8 rounds
model = SubsidyModel(n_agents=500, strategy='cognitive', seed=42)
results_list = []
for r in range(8):
    model.step()
    r_result = model.collect_results()
    results_list.append(r_result)
    print(f'Round {r+1}: ROI={r_result["roi"]:.2f}, ΔGTV={r_result["delta_gtv"]:.1f}, Coverage={r_result["coverage"]:.2%}')

sim_df = pd.DataFrame(results_list)
print(f'\nMean ROI: {sim_df["roi"].mean():.2f}')
print(f'Total ΔGTV: {sim_df["delta_gtv"].sum():.1f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(sim_df.index + 1, sim_df['roi'], 'o-', color='#2ecc71')
axes[0].set_title('ROI per Round')
axes[0].set_ylabel('ROI')

axes[1].plot(sim_df.index + 1, sim_df['delta_gtv'].cumsum(), 'o-', color='#3498db')
axes[1].set_title('Cumulative ΔGTV')
axes[1].set_ylabel('ΔGTV (¥)')

axes[2].plot(sim_df.index + 1, sim_df['coverage'], 'o-', color='#e74c3c')
axes[2].set_title('Coverage Rate')
axes[2].set_ylabel('Coverage')

for ax in axes:
    ax.set_xlabel('Round')
    ax.set_xticks(range(1, 9))

plt.suptitle('Mesa ABM: Cognitive Agent Strategy (500 agents, 8 rounds)', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('../output/notebook_mesa_abm.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Multi-World Evaluation

**Key innovation**: Run parallel simulation worlds with different subsidy strategies, then compare outcomes. This decouples assumption risk from random noise.

In [ ]:
from src.simulation.mesa_agent_model import MultiWorldModel

mw = MultiWorldModel(n_agents=300, n_rounds=8, seed=42)
world_results = mw.run_all_strategies()
comparison = mw.compare_worlds()

print('Multi-World Comparison:')
print(comparison.to_string(index=False))


In [ ]:
# Visualize multi-world comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

strategies = comparison['strategy'].tolist()
x_pos = range(len(strategies))
colors = ['#95a5a6', '#e74c3c', '#3498db', '#2ecc71']

axes[0].bar(x_pos, comparison['avg_roi'], color=colors[:len(strategies)])
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(strategies, rotation=15)
axes[0].set_title('Average ROI')
axes[0].set_ylabel('ROI')

axes[1].bar(x_pos, comparison['cumulative_delta_gtv'], color=colors[:len(strategies)])
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(strategies, rotation=15)
axes[1].set_title('Cumulative ΔGTV')
axes[1].set_ylabel('ΔGTV (¥)')

axes[2].bar(x_pos, comparison['avg_coverage'], color=colors[:len(strategies)])
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(strategies, rotation=15)
axes[2].set_title('Average Coverage')
axes[2].set_ylabel('Coverage')

plt.suptitle('Multi-World Strategy Comparison', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('../output/notebook_multi_world.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Social Network Contagion

Modeling how coupon redemption behavior spreads through social networks using SIR contagion.

In [ ]:
from src.simulation.network_contagion import SocialNetwork, SocialContagion

# Build network
sn = SocialNetwork()
G = sn.build_barabasi_albert(n=200, m=3, seed=42)
print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Run contagion
sc = SocialContagion()
seed_nodes = list(range(10))
result = sc.propagate(G, seed_nodes, contagion_rate=0.2, n_steps=30, seed=42)
sir_ts = result['time_series']
print(f'Cascade size: {result["cascade_size"]} nodes infected')

# Plot SIR curves
fig, ax = plt.subplots(figsize=(10, 5))
s_data = [s['S'] for s in sir_ts]
i_data = [s['I'] for s in sir_ts]
r_data = [s['R'] for s in sir_ts]
ax.plot(s_data, label='Susceptible', color='#3498db')
ax.plot(i_data, label='Infected (coupon adopters)', color='#e74c3c')
ax.plot(r_data, label='Recovered (immune)', color='#2ecc71')
ax.set_xlabel('Time Step')
ax.set_ylabel('Number of Nodes')
ax.set_title('SIR Contagion on Barabási–Albert Network\n(Coupon redemption spillover)')
ax.legend()
plt.tight_layout()
plt.savefig('../output/notebook_sir_contagion.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Evaluation Metrics

Bootstrap confidence intervals, E-value sensitivity analysis, and multi-world robustness metrics.

In [ ]:
from src.evaluation.metrics import bootstrap_ci, compute_roi, e_value

# Bootstrap CI
outcome = causal_data['outcome'].values
ci = bootstrap_ci(outcome, statistic=np.mean, n_bootstrap=1000, ci=0.95)
print(f'Bootstrap 95% CI for mean outcome: [{ci[0]:.4f}, {ci[1]:.4f}]')

# ROI
roi = compute_roi(subsidy_cost=100, incremental_gtv=250)
print(f'ROI: {roi:.2f}')

# E-value (sensitivity to unmeasured confounding)
ev = e_value(rr=2.0)
print(f'E-value for RR=2.0: {ev:.4f}')
print(f'  Interpretation: An unmeasured confounder would need to have a risk ratio of ≥{ev:.2f}')
print(f'  with both treatment and outcome to explain away the observed effect.')


## Summary

| Component | Key Result |
|-----------|-------------|
| **CausalML** | T/X/DR-Learner CATE correlations > 0.84 with true CATE |
| **DoWhy** | ATE ≈ 2.0 (true ATE), with refutation validation |
| **Cognitive Agents** | Different mental account types → distinct redemption patterns |
| **Mesa ABM** | Cognitive strategy achieves higher ROI than static/dynamic/random |
| **Multi-World** | Robust policy comparison across parallel simulation worlds |
| **Network Contagion** | SIR model captures social spillover of coupon adoption |
| **Evaluation** | Bootstrap CI + E-value for robustness checks |